# Calculate Node statistics

> In some cases it may be valuable to assign node properteis based on their location in the graph. For example, one might wish for the nodes closest to the prediciton node to have more units than those further away.

In [ ]:
# import re
from sparsevnn.core import *
# import torch
# from torch import nn 
# import numpy as np
# import plotly.express as px

In [ ]:
graph_connections = {
 'A': ['100278565'],
 'B': ['100278565'],
 'C': ['100383860'],
 'D': ['B', 'C'],
 'y_hat': ['A', 'C', 'D']}

dict_to_digraph(edge_dict = graph_connections)

NameError: name 'dict_to_digraph' is not defined

In [ ]:
graph_connections

In [ ]:
# consider only one pair of nodes
def vertex_subsequent(
    edge_dict,
    start,
    end
):
    "Count the number of vertices between two nodes (unidirectional)"
    nos = (start not in edge_dict.keys())
    noe = (end not in edge_dict.keys())
    if nos & noe:
        raise Exception('`start` and `end` nodes are not in `edge_dict`')         
    elif nos:
        raise Exception('`start` node is not in `edge_dict`')
    elif noe:
        raise Exception('`end` nodes is not in `edge_dict`')
    else:
        query_list = edge_dict[end]
        for i in range(len(edge_dict.keys())):
            if start in query_list:
                return i
            else:
                query_list = sum([edge_dict[e] for e in [ee for ee in query_list if ee in edge_dict.keys()]], [])
        return None

Note that because this function only checks for along subsequent verticies, if we swap the start and end or point to the same node twice we don't get a distance.

In [ ]:
[
    vertex_subsequent(
        edge_dict = graph_connections,
        start = 'B',
        end   = 'y_hat'
    ), vertex_subsequent(
        edge_dict = graph_connections,
        start = 'y_hat',
        end   = 'B'
    ), vertex_subsequent(
        edge_dict = graph_connections,
        start = 'y_hat',
        end   = 'y_hat'
    )]

This function we can extend trivially to get the distance between to nodes.

In [ ]:
def vertex_between(
    edge_dict,
    node0,
    node1
):
    "Count the number of vertices between two nodes"
    res = vertex_subsequent(
        edge_dict = edge_dict,
        start = node0,
        end =node1)
    if res == None:
        res = vertex_subsequent(
            edge_dict = edge_dict,
            start = node1,
            end =node0)
    return res 

Now because the number of verticies between is communicative the order of nodes doesnt matter.

In [ ]:
vertex_between(
    edge_dict = graph_connections,
    node0 = 'B',
    node1 ='y_hat'

) == vertex_between(
    edge_dict = graph_connections,
    node0 ='y_hat',
    node1 = 'B'

)

In some cases it will be better to build an index to begin with. 

In [ ]:
def vertex_from_end(
        edge_dict,
        end 
        ):
    "build a reference for all child nodes of a given root"

    # look at edge_dict's entries for all the nodes in a query list. Add them to dict out with value i and return new values and new i (distance from node)
    def _temp(i, query_list):
        new_queries = []
        for e in query_list:
            if e not in edge_dict.keys():
                pass
            else:
                new_queries += edge_dict[e]
                for ee in edge_dict[e]:
                    if ee not in out.keys():
                        out[ee] = i
                    elif out[ee] > i:
                        out[ee] = i
                    else:
                        # entry exists but is not the shortest path
                        pass 
        return new_queries, i+1

    out = {end:0}
    i = 1 
    ql = [end]
        
    # instead of using a while and checking for the base case of an empty list, I'll assume the worst case all nodes are sequential
    for j in range(len(edge_dict.keys())):
        if ql == []:
            break
        else:
            ql, i = _temp(i = i, query_list = ql)

    return out

In [ ]:
dist = vertex_from_end(
        edge_dict = graph_connections,
        end ='y_hat')
dist

... and once we have a node to distance dictionary we can use a comprehension to create the reverse lookup. 

In [ ]:
{i:[ee for ee in dist if dist[ee] == i] for i in set([dist[e] for e in dist])}

With the index complete we can check for 

In [ ]:
# The same graph, plus a little renaming wizardry
new_names = {e: e+'\ndist='+str(dist[e]) for e in dist}

dict_to_digraph(edge_dict = 
                {new_names[e]: [new_names[ee] for ee in graph_connections[e]] 
                 for e in graph_connections}
                )